In [1]:
import cv2
from ultralytics import YOLO

In [2]:
input_video = '../inputs/road.mp4'
haar_output = '../outputs/haar_video.mp4'

car_detector = cv2.CascadeClassifier('cars.xml')

if car_detector.empty():
    print("Error: Failed to load cascade file. Please verify the file path.")
    exit()

video_capture = cv2.VideoCapture(input_video)

if not video_capture.isOpened():
    print("Error: Unable to open input video.")
    exit()

frame_rate = int(video_capture.get(cv2.CAP_PROP_FPS))
frame_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
codec = cv2.VideoWriter_fourcc(*'mp4v')
haar_writer = cv2.VideoWriter(haar_output, codec, frame_rate, (frame_width, frame_height))

while True:
    success, video_frame = video_capture.read()
    if not success:
        print("Video processing completed.")
        break

    gray_frame = cv2.cvtColor(video_frame, cv2.COLOR_BGR2GRAY)

    detected_cars = car_detector.detectMultiScale(
        gray_frame,
        scaleFactor=1.2,
        minNeighbors=4,
        minSize=(40, 40)
    )

    for (x, y, w, h) in detected_cars:
        cv2.rectangle(video_frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
        cv2.putText(
            video_frame,
            "Car",
            (x, y - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 0, 0),
            2
        )

    haar_writer.write(video_frame)

video_capture.release()
haar_writer.release()

print(f"Video processed using Haar Cascade and saved as {haar_output}")

Video processing completed.
Video processed using Haar Cascade and saved as ../outputs/haar_video.mp4


In [3]:
source_video = '../inputs/road.mp4'
yolo_output = '../outputs/yolo_video.mp4'

object_detector = YOLO('yolov8x.pt')

video_reader = cv2.VideoCapture(source_video)

if not video_reader.isOpened():
    print("Error: Could not load source video.")
    exit()

video_fps = int(video_reader.get(cv2.CAP_PROP_FPS))
video_width = int(video_reader.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_reader.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_codec = cv2.VideoWriter_fourcc(*'mp4v')
yolo_writer = cv2.VideoWriter(yolo_output, video_codec, video_fps, (video_width, video_height))

while True:
    success, current_frame = video_reader.read()
    if not success:
        print("YOLO detection completed.")
        break

    yolo_results = object_detector(current_frame)

    for detection in yolo_results[0].boxes:
        x_start, y_start, x_end, y_end = detection.xyxy[0].numpy()
        confidence = detection.conf[0].item()
        class_id = int(detection.cls[0].item())
        detected_label = object_detector.names[class_id]

        cv2.rectangle(current_frame, (int(x_start), int(y_start)), (int(x_end), int(y_end)), (0, 0, 255), 2)
        cv2.putText(
            current_frame,
            f'{detected_label} {confidence:.2f}',
            (int(x_start), int(y_start) - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 255),
            2
        )

    yolo_writer.write(current_frame)

video_reader.release()
yolo_writer.release()

print(f"YOLO-based processed video saved as {yolo_output}")


0: 384x640 10 cars, 1 truck, 293.7ms
Speed: 3.1ms preprocess, 293.7ms inference, 6.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 2 trucks, 297.0ms
Speed: 1.2ms preprocess, 297.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 1 truck, 255.1ms
Speed: 0.8ms preprocess, 255.1ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 1 truck, 263.9ms
Speed: 0.8ms preprocess, 263.9ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 2 trucks, 262.4ms
Speed: 0.9ms preprocess, 262.4ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 2 trucks, 322.3ms
Speed: 1.3ms preprocess, 322.3ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 2 trucks, 310.3ms
Speed: 0.8ms preprocess, 310.3ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 2 trucks, 269.8ms
Speed: